In [3]:
%pip install scikit-learn

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl (8.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [sci

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

In [2]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target

In [3]:
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [4]:
train_df , test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

In [5]:
x_train , y_train = train_df.drop(columns=['target']), train_df['target']
x_test , y_test = test_df.drop(columns=['target']), test_df['target']

In [6]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [7]:
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

In [19]:
class IrisClassifier(nn.Module):
   def __init__(self, input_dim, hidden_dim, output_dim):
      super(IrisClassifier, self).__init__()
      self.Network = nn.Sequential(
         nn.Linear(input_dim, hidden_dim),
         nn.ReLU(),
         nn.Linear(hidden_dim, hidden_dim),
         nn.ReLU(),
         nn.Linear(hidden_dim, output_dim)
      )

   def forward(self, x):
         return self.Network(x)

In [20]:
input_dim = x_train.shape[1]
hidden_dim = 16
output_dim = 3

In [21]:
model = IrisClassifier(input_dim, hidden_dim, output_dim)

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [23]:
epochs = 500

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predication  = model(x_train_tensor)
    loss = criterion(predication, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}')




Epoch [50/500], Loss: 0.0579
Epoch [100/500], Loss: 0.0328
Epoch [150/500], Loss: 0.0299
Epoch [200/500], Loss: 0.0276
Epoch [250/500], Loss: 0.0255
Epoch [300/500], Loss: 0.0119
Epoch [350/500], Loss: 0.0035
Epoch [400/500], Loss: 0.0015
Epoch [450/500], Loss: 0.0009
Epoch [500/500], Loss: 0.0006


In [24]:
model.eval()
with torch.no_grad():
    y_pred = model(x_test_tensor)
    y_pred_labels = torch.argmax(y_pred, dim=1)

    accuracy = (y_pred_labels == y_test_tensor).sum().item() / y_test_tensor.size(0)
    print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.9667


In [26]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [25]:
def prdict_iris(sepal_length, sepal_width, petal_length, petal_width):
    input_data = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    input_data_scaled = scaler.transform(input_data)
    input_tensor = torch.tensor(input_data_scaled, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        prediction = model(input_tensor)
        predicted_class = torch.argmax(prediction, dim=1).item()
    return predicted_class


In [27]:
predicted_class = prdict_iris(5.1, 3.5, 1.4, 0.2)
print(f'Predicted class: {predicted_class}, Class name: {iris.target_names[predicted_class]}')

Predicted class: 0, Class name: setosa


c:\Users\Ali\miniconda3\envs\myproject\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
